# Run the coupled MODFLOW + D-Flow FM + SWMM models

Scenario-driven runner for the Greenport coupled models. A *scenario* is one
combination of:

- **D-Flow FM discretization** (`resolution`: `coarse` / `medium` / `high`),
- **MODFLOW &harr; D-Flow FM coupling frequency** (`mf_couple_freq_hours`), and
- **number of SWMM &harr; MODFLOW connections** (`n_junctions`, capped to the
  junctions actually available).

For each scenario the notebook:
1. selects the D-Flow FM grid via `liss_settings.get_dflow_control_path`;
2. resolves the SWMM&ndash;MODFLOW connections and rebuilds the MODFLOW SWMM well
   package (`gwf_swmm.wel`);
3. copies the matching sewer tracer file
   `data/GP/Sewer_sourcesink_n{n_connections}__{tag}.bc` into the D-Flow FM run
   directory;
4. runs the three models coupled with a variable D-Flow-FM time step;
5. saves MODFLOW, SWMM, and D-Flow tracer results to a **scenario-named**
   directory for the step3 plotting notebooks; and
6. regenerates the sewer tracer `.bc` from this run's SWMM results.

> Derived from `step2_run_dflow-modflow_variable_dt_BNB.ipynb` and
> `step2_run_dflow-modflow_variable_dt.ipynb`.

## Imports

In [ ]:
import os

# Three engines in one process means the Intel OpenMP runtime gets loaded twice:
# numpy/MKL loads libiomp5md.dll from the conda env at import, and MODFLOW resolves
# the SAME dll again out of the D-Flow FM dll directory that is prepended to PATH
# before dflowfm is initialized. MF6 then aborts the whole process with
# "OMP: Error #15: ... already initialized" on the first coupling step.
#
# The two files are byte-identical (same sha256, version 20240320), so this is a
# duplicate *load path*, not a mix of different OpenMP runtimes -- the "may silently
# produce incorrect results" caveat in the OMP message is about the latter. Must be
# set before numpy is imported, hence the position at the top of this cell.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import time
import datetime
import shutil
import sys
import pathlib as pl

import numpy as np
import pandas as pd
import geopandas as gpd
import xugrid

import flopy
from modflowapi import ModflowApi
from modflowapi.extensions import ApiSimulation
from bmi.wrapper import BMIWrapper

import pyswmm
from pyswmm import Simulation, Nodes, Output

# pyswmm 2.x keeps the output-attribute enums in swmm.toolkit, not pyswmm.output
from swmm.toolkit.shared_enum import NodeAttribute

In [ ]:
sys.path.append("../common")
from liss_settings import (
    libmf6,
    get_dflow_control_path,
    get_dflow_grid_name,
    get_dflow_dtuser,
    get_modflow_grid_name,
    get_modflow_coupling_tag,
    get_scenario_name,
    get_results_path,
    silent,
    verbosity,
)
from swmm_mf_connect import intersect_points_grid

# D-Flow FM initialize() changes the working directory; capture the notebook dir
# so the post-run relative (../data) paths resolve correctly.
nb_dir = pl.Path.cwd().resolve()

## Scenario configuration

**Edit these values to define the scenario.** Everything downstream (tags,
tracer file, output directory, well count, conductance scaling) is derived from
them &mdash; keyed on the *actual* number of connections resolved below.

In [ ]:
domain = "gp"                     # model domain
boundary_condition = "chd"        # MODFLOW coastal BC: "chd" or "ghb"
resolution = "coarse"             # D-Flow FM discretization: "coarse" | "medium" | "high"
mf_couple_freq_hours = 8.0        # MODFLOW <-> D-Flow FM coupling frequency (hours)
n_junctions = 500                 # requested # of SWMM <-> MODFLOW connections
                                  # (capped to the junctions actually available)

# Smoke test. When set to a number of days, the D-Flow FM computation is shortened
# to that many days from StartDateTime -- on the RUN COPY of the .mdu only, never on
# the base model -- and the scenario name is tagged so a short run cannot overwrite
# full-run results. MODFLOW and SWMM keep their full windows: the coupling loop is
# driven by the D-Flow clock, so both are simply stopped early and finalized.
# None = the full run. A 2-day smoke test is ~0.75 min / 6 coupling steps; the full
# 89-day coarse window is ~267 coupling steps.
smoke_test_days = None

# ---- SWMM <-> MODFLOW exchange conductance ----------------------------------
# None  -> legacy behaviour: a per-connection CONSTANT (0.001 ft2/d infiltration,
#          0.0002 exfiltration) rescaled by n_original/n_connections, so the total
#          is fixed at n_original*0.001 = 0.012 ft2/d no matter how much pipe each
#          connection stands for.
# float -> leakance in 1/d applied to each junction's own leakage AREA,
#          C_j = leakance * pi * D_j * L_j, with L_j half of every conduit it
#          touches. Served length varies ~24x across these junctions (28-677 ft),
#          which a constant cannot represent.
#
# Reference points for the total conductance over this network (125,739 ft2 of
# pipe wall, 56,638 ft of pipe):
#     legacy constant          -> 0.012 ft2/d   (equivalent leakance 9.5e-8 1/d)
#     100 gpd/in-dia/mile      -> 363  ft2/d    (leakance 0.0029 1/d,  ~3% of outfall)
#     500 gpd/in-dia/mile      -> 1815 ft2/d    (leakance 0.0144 1/d, ~16% of outfall)
# 100-500 gpd/in/mi is the usual allowable-infiltration band for gravity sanitary
# sewers, so the legacy constant sits about five orders of magnitude below it.
pipe_leakance = None

# Exfiltration is harder than infiltration: the legacy coefficients were 0.001 vs
# 0.0002, a 5:1 ratio, and that asymmetry is kept here. Used only when
# pipe_leakance is set.
exfiltration_ratio = 0.2

# Pipe water depth below this (ft) counts as a dry pipe: the driving head then
# falls back to the invert, and a dry pipe with groundwater below the invert
# exchanges nothing at all.
PIPE_DEPTH_MIN = 0.01
# -----------------------------------------------------------------------------

# SWMM input model. This MUST be the same model that
# swmm_mf_connect.intersect_points_grid() resolves junctions against -- it defaults
# to ../swmm/{domain}/{domain}_sewer.inp. gp_sewer.inp is the full network (251
# junctions, 245 conduits, 69 subcatchments); greenport_detailedsewer_v4_nosub175.inp
# is a 154-junction subset of it with the subcatchments stripped, and its junctions
# are named without zero padding ("1" vs "001"), so mixing the two makes every
# junction lookup fail.
swmm_inp_name = "gp_sewer.inp"

# SWMM outfall whose total inflow drives the D-Flow FM [SourceSink] tracer.
#   gp_sewer.inp                          -> "O3"  ("171" exists but is a junction)
#   greenport_detailedsewer_v4_nosub175   -> "171"
outfall_node = "O3"

# The SWMM -> MODFLOW inflow coefficients in update_swmm were calibrated for this
# many connections; the per-connection coefficient is rescaled by
# (n_original / n_connections) so the TOTAL exchange conductance is constant.
n_original = 12

### Derived grid / coupling configuration

In [ ]:
control_path = get_dflow_control_path(domain, resolution)
dflow_grid_name = get_dflow_grid_name(control_path)
dflowfm_dtuser = get_dflow_dtuser(control_path)
mf_grid_name = get_modflow_grid_name(domain=domain, boundary_condition=boundary_condition)
mf_tag = get_modflow_coupling_tag(mf_couple_freq_hours)

mf_couple_freq = mf_couple_freq_hours * 60.0 * 60.0
dflow_per_mf = int(mf_couple_freq / dflowfm_dtuser)      # D-Flow steps per MODFLOW step
mf_couple_nstp = int(86400.0 / (dflow_per_mf * dflowfm_dtuser))

print(f"D-Flow grid   : {dflow_grid_name}  (DtUser={dflowfm_dtuser}s)")
print(f"MODFLOW grid  : {mf_grid_name}")
print(f"coupling tag  : {mf_tag}  ({mf_couple_freq_hours} h, {dflow_per_mf} D-Flow steps / MF step)")

### Unit conversions and coupling constants

In [ ]:
d2sec = 24.0 * 60.0 * 60.0
hrs2sec = 60.0 * 60.0
m2ft = 3.28081
cfd2cms = 1.0 / ((m2ft**3) * 86400.0)

HDRY = -1e30
DEPTH_MIN = 0.1

str(libmf6), libmf6.is_file()

## SWMM &harr; MODFLOW connections (resolved first)

`intersect_points_grid` samples `n_junctions` SWMM junctions, intersects them
with the MODFLOW grid, and returns the (layer, row, column) cell for each. It
**caps the request to the junctions actually available**, so the returned
`n_connections` is the actual number of connections &mdash; and that value drives
the scenario name, the tracer-bc lookup, the well package, and the conductance
scaling.

In [ ]:
(n_connections, junctions, mf6_cells, swmm_inverts, possible_junctions) = intersect_points_grid(
    domain=domain,
    boundary_condition=boundary_condition,
    n_junctions=n_junctions,
)
print(f"requested {n_junctions} -> {n_connections} actual SWMM <-> MODFLOW connections")
print(f"unique MODFLOW cells: {len(set(mf6_cells.values()))}")

# The sjoin inside intersect_points_grid can emit more than one row for a junction
# that lands on a cell boundary; mf6_cells is a dict, so those collapse silently
# while the returned n_connections is the pre-join count. Everything downstream
# (scenario name, tracer bc lookup, MAXBOUND, conductance_scale) keys on
# n_connections, so make sure the two agree before building anything.
assert len(mf6_cells) == n_connections, (
    f"n_connections={n_connections} but len(mf6_cells)={len(mf6_cells)} - a junction "
    "probably intersected more than one grid cell"
)
assert set(junctions) == set(mf6_cells), "junction list and mf6_cells keys disagree"

# rescale the inflow coefficients so the TOTAL conductance is constant
conductance_scale = n_original / n_connections
print(f"conductance scale = n_original/n_connections = {n_original}/{n_connections} = {conductance_scale:.5f}")


def _swmm_sections(inp_path, name):
    """Yield the whitespace-split data rows of one SWMM .inp section."""
    rows, inside = [], False
    for line in pl.Path(inp_path).read_text(errors="replace").splitlines():
        s = line.strip()
        if s.startswith("["):
            inside = s.upper().startswith(f"[{name}")
            continue
        if inside and s and not s.startswith(";"):
            rows.append(s.split())
    return rows


def pipe_leakage_conductance(inp_path, names, leakance):
    """Per-junction leakage conductance in ft2/d from the sewer geometry.

    C_j = leakance * (pi * D * L)_j, where each junction is credited with HALF the
    length of every conduit it touches. This replaces a per-connection constant,
    which cannot be right dimensionally: served pipe length varies ~24x across
    these junctions (28 to 677 ft), so a constant makes a junction representing
    677 ft of pipe leak the same as one representing 28 ft.

    gp_sewer.inp declares FLOW_UNITS CMS, so its lengths and diameters are metres;
    MODFLOW is in feet, hence the m2ft conversions.
    """
    conduits = {r[0]: (r[1], r[2], float(r[3]))
                for r in _swmm_sections(inp_path, "CONDUITS") if len(r) >= 4}
    diam = {r[0]: float(r[2]) for r in _swmm_sections(inp_path, "XSECTIONS")
            if len(r) >= 3 and r[1].lower().startswith("circ")}
    area_m2 = {}
    for link, (a, b, length_m) in conduits.items():
        d = diam.get(link)
        if d is None:
            continue
        for node in (a, b):
            area_m2[node] = area_m2.get(node, 0.0) + 0.5 * np.pi * d * length_m
    missing = [j for j in names if j not in area_m2]
    assert not missing, f"{len(missing)} junctions have no conduit geometry: {missing[:5]}"
    return {j: leakance * area_m2[j] * m2ft**2 for j in names}


if pipe_leakance is None:
    swmm_conductance = None
    print("SWMM exchange: legacy constant coefficient (0.001 / 0.0002 ft2/d per "
          f"connection, scaled by {conductance_scale:.5f})")
else:
    swmm_path_for_geom = (pl.Path.cwd() / f"../swmm/{domain}/{swmm_inp_name}").resolve()
    swmm_conductance = pipe_leakage_conductance(swmm_path_for_geom, junctions, pipe_leakance)
    _c = np.array(list(swmm_conductance.values()))
    print(f"SWMM exchange: area-based, leakance={pipe_leakance:g} 1/d")
    print(f"  conductance ft2/d: min={_c.min():.3f} median={np.median(_c):.3f} "
          f"max={_c.max():.3f} total={_c.sum():,.1f}")
    print(f"  legacy total was {n_original * 0.001:.4f} ft2/d "
          f"({_c.sum() / (n_original * 0.001):,.0f}x smaller)")

### Scenario name and output location

In [ ]:
# The scenario id and results location come from liss_settings so that the step3
# plotting notebooks resolve exactly the same paths from the same four knobs --
# there is no second copy of this naming rule to drift out of sync.
scenario = get_scenario_name(domain, resolution, mf_couple_freq_hours, n_connections)
results_ws = get_results_path(domain, resolution, mf_couple_freq_hours, n_connections)

if smoke_test_days:
    # keep short runs in their own results / run directories
    scenario += f"_smoke{smoke_test_days:g}d"
    results_ws = results_ws.parent / scenario

results_ws.mkdir(parents=True, exist_ok=True)
print("scenario   :", scenario)
print("results ->  ", results_ws)

### Select the sewer tracer file for this scenario

The D-Flow FM `[SourceSink]` tracer forcing is a pre-tabulated SWMM outflow,
named by the actual number of connections and the coupling tag. It is copied
into the D-Flow FM run directory (as `Sewer_sourcesink.bc`, matching the
`FlowFM_bnd.ext` reference) before D-Flow FM is initialized.

In [ ]:
# The tracer forcing is SCENARIO-specific. The SWMM outflow depends on the MODFLOW
# heads, which depend on the D-Flow grid and the coupling frequency, so each
# (resolution, n_connections, coupling tag) combination converges to its own sewer
# series. The authoritative copy of each lives in data/<DOMAIN>/ and is rewritten
# at the end of this notebook; re-running the same scenario picks its own file back
# up, which is the iteration that makes the coupling self-consistent.
#
# The resolution is part of the name: without it a midres run would read and then
# overwrite the coarse file.
tracer_dir = (nb_dir / f"../data/{domain.upper()}").resolve()
tracer_bc_scenario = tracer_dir / (
    f"Sewer_sourcesink_{resolution}_n{n_connections:03d}__{mf_tag}.bc"
)
# First run of a scenario: fall back to the resolution-agnostic seed written by
# data/GP/update_files.py from a standalone SWMM run.
tracer_bc_seed = tracer_dir / f"Sewer_sourcesink_n{n_connections:03d}__{mf_tag}.bc"

if tracer_bc_scenario.is_file():
    tracer_bc_src = tracer_bc_scenario
    print(f"tracer forcing : {tracer_bc_src.name}  (previous run of this scenario)")
else:
    tracer_bc_src = tracer_bc_seed
    print(f"tracer forcing : {tracer_bc_src.name}  (seed - first run of this scenario)")

print(tracer_bc_src, tracer_bc_src.is_file())
assert tracer_bc_src.is_file(), (
    f"no tracer bc for {resolution} / n{n_connections:03d} / {mf_tag}:\n"
    f"  scenario: {tracer_bc_scenario}\n"
    f"  seed    : {tracer_bc_seed}\n"
    f"regenerate the seed with data/{domain.upper()}/update_files.py first."
)

## D-Flow FM &rarr; MODFLOW mapping weights

Produced by `step1a` (GHB) and `step1b` (CHD), keyed on the D-Flow + MODFLOW grid
names.

In [ ]:
ghb_map = pl.Path(f"../mapping/{domain}/dflow_{dflow_grid_name}_to_{mf_grid_name}_ghb.npz")
chd_map = pl.Path(f"../mapping/{domain}/dflow_{dflow_grid_name}_to_{mf_grid_name}_chd.npz")
if not ghb_map.is_file():
    ghb_map = pl.Path(f"../mapping/dflow_{dflow_grid_name}_to_{mf_grid_name}_ghb.npz")
if not chd_map.is_file():
    chd_map = pl.Path(f"../mapping/dflow_{dflow_grid_name}_to_{mf_grid_name}_chd.npz")
print(ghb_map, ghb_map.is_file())
print(chd_map, chd_map.is_file())

npz = np.load(ghb_map)
dflow2mfghb, ghbmask, ghb2qext = npz["dflow2mfghb"], npz["ghbmask"], npz["ghb2qext"]
npz = np.load(chd_map)
dflow2mfchd, chdmask, chd2qext = npz["dflow2mfchd"], npz["chdmask"], npz["chd2qext"]
print("ghb:", dflow2mfghb.shape, "chd:", dflow2mfchd.shape)

## Load the base MODFLOW model into the scenario run directory

In [ ]:
mf_base_path = pl.Path(f"../modflow/{mf_grid_name}/base/").resolve()
mf_run_path = pl.Path(f"../modflow/{mf_grid_name}/run_{scenario}/").resolve()

sim = flopy.mf6.MFSimulation.load(sim_ws=mf_base_path, verbosity_level=verbosity())
gwf = sim.get_model()
sim.set_sim_path(mf_run_path)

# Reset the run directory the same way the D-Flow FM one is reset below. flopy
# only ever WRITES into this directory, so without a wipe files from an earlier
# scenario configuration survive alongside the current ones -- e.g. a previous
# gwf_0.wel sitting next to gwf_swmm.wel, unreferenced by gwf.nam but very
# confusing to read. Note this also removes the previous run's gwf.hds / gwf.cbc /
# gwt.ucn, which step3 reads from here; they are rewritten by this run, so only a
# run that fails partway leaves you without them.
#
# Hard stops before the rmtree, mirroring the D-Flow guards: never the base model,
# never a parent of it, and the name must actually look like a run directory.
assert mf_run_path != mf_base_path, f"run dir must not be the base dir: {mf_base_path}"
assert mf_base_path not in mf_run_path.parents, "run dir must not contain base"
assert mf_run_path.name.startswith("run_"), (
    f"refusing to delete {mf_run_path}: name does not start with 'run_'"
)

if mf_run_path.is_dir():
    shutil.rmtree(mf_run_path)
(mf_run_path / "outputs").mkdir(parents=True, exist_ok=True)

print("MODFLOW base   :", mf_base_path)
print("MODFLOW run dir:", mf_run_path)

### Set the MODFLOW time steps per stress period for the coupling frequency

In [ ]:
tdis = sim.get_package("TDIS")
perioddata = tdis.perioddata.array
perioddata["nstp"] = mf_couple_nstp
tdis.perioddata = perioddata

### Rebuild the SWMM well package from the connections

One WEL entry per SWMM connection (from `mf6_cells`), so the number of wells
matches the number of connections for this scenario. Multiple junctions may land
in one cell; MODFLOW sums their flux.

In [ ]:
# The base model already registers a SWMM well package (gwf_swmm.wel, MAXBOUND 17,
# boundnames j1..). Drop it so this scenario's package is the only WEL6 entry named
# SWMM in the name file.
if gwf.get_package("SWMM") is not None:
    gwf.remove_package("SWMM")

# MF6 resolves an OBS "ID" field as a cellid FIRST and only falls back to a
# boundname if it does not parse as numbers. The SWMM junctions are named "123",
# "003", ... so a bare name is read as a node number and MF6 aborts during
# initialize() with "Cell number cannot be determined in line '123'" -- killing the
# host process, since MF6 stops rather than raising. The base model prefixes its
# boundnames ('j1', 'j2', ...) for exactly this reason. The prefix is injective, so
# strip the leading "j" to map an obs column back to its junction.
BOUNDNAME_PREFIX = "j"

swmm_well_spd = []
swmm_well_obs = []
for name, (lay, row, col) in mf6_cells.items():
    # boundnames=True -> rows are [cellid, q, boundname]. The observations below key
    # on those boundnames, so they have to actually be written or MF6 rejects the OBS.
    bname = f"{BOUNDNAME_PREFIX}{name}"
    swmm_well_spd.append([(lay, row, col), 0.0, bname])
    swmm_well_obs.append((bname, "WEL", bname))

assert len({b for b, _, _ in swmm_well_obs}) == len(swmm_well_obs), "duplicate boundnames"
assert not any(b.lstrip("-+").isdigit() for b, _, _ in swmm_well_obs), (
    "a boundname is still purely numeric and MF6 would read it as a cellid"
)

# Without an explicit filename flopy names the package after the model and a
# counter (gwf_0.wel), which does not match the gwf_swmm.wel the base model used
# and makes the scenario package hard to pick out in the run directory.
swmm_wel_filename = "gwf_swmm.wel"

wel_swmm = flopy.mf6.ModflowGwfwel(
    gwf,
    save_flows=True,
    boundnames=True,
    stress_period_data={0: swmm_well_spd},
    observations={"outputs/swmm_well_obs.csv": swmm_well_obs},
    pname="SWMM",
    filename=swmm_wel_filename,
)
sim.write_simulation(silent=silent())

assert (mf_run_path / swmm_wel_filename).is_file(), (
    f"{swmm_wel_filename} was not written to {mf_run_path}"
)
print(f"SWMM well package: {len(swmm_well_spd)} entries in {len(set(mf6_cells.values()))} cells")
print(f"  -> {swmm_wel_filename}")
print("  (MF6 sums WEL entries that share a cell, so co-located connections accumulate)")

### Base GHB / CHD stress-period data (templates used each step)

In [ ]:
ghb_data0 = gwf.ghb.stress_period_data.get_dataframe()[0]
assert ghb_data0.shape[0] == ghbmask.shape[0]

chd_surface = gwf.get_package("chd_surface")
chd_data0 = chd_surface.stress_period_data.get_dataframe()[0]
assert chd_data0.shape[0] == chdmask.shape[0]

## Set up and initialize D-Flow FM

In [ ]:
dflow_dirpath = (control_path.parent.parent.parent / "dflowfm_dll.2026.01").resolve()
dflow_base = control_path.parent.resolve()

# Scenario-named run directory, as a SIBLING of the base scenario dir. The base model
# is only ever read from and copied out of, and two scenarios (or two resolutions)
# can never clobber each other. Matches the "run*/" rule already in .gitignore.
dflow_working = (dflow_base.parent / f"run_{scenario}").resolve()
dflow_config = dflow_working / "FlowFM.mdu"

# hard stop before the rmtree below if the paths ever collapse onto each other
assert dflow_working != dflow_base, f"run dir must not be the base dir: {dflow_base}"
assert dflow_base not in dflow_working.parents, "run dir must not be inside base"

# reset the working (run) copy from base, skipping output/run leftovers in base
if dflow_working.is_dir():
    shutil.rmtree(dflow_working)
shutil.copytree(
    dflow_base,
    dflow_working,
    ignore=shutil.ignore_patterns("output", "run_*", "*.dia"),
)
(dflow_working / "output").mkdir(parents=True, exist_ok=True)

# copy the scenario tracer file in as the source/sink bc referenced by FlowFM_bnd.ext
shutil.copyfile(tracer_bc_src, dflow_working / "Sewer_sourcesink.bc")
print("D-Flow base    :", dflow_base)
print("D-Flow run dir :", dflow_working)

# Smoke test: shorten the computation by rewriting StopDateTime in the RUN COPY of
# the .mdu. dflow_working was just recreated from dflow_base above, so the base .mdu
# is never touched and the edit cannot accumulate across runs.
if smoke_test_days:
    _dt_fmt = "%Y%m%d%H%M%S"
    _lines = dflow_config.read_text().splitlines(keepends=True)

    def _mdu_raw(tag):
        for _ln in _lines:
            if _ln.startswith(tag):
                return _ln.split("=", 1)[1].split("#")[0].strip()
        return None

    _start = datetime.datetime.strptime(_mdu_raw("StartDateTime"), _dt_fmt)
    _stop_full_raw = _mdu_raw("StopDateTime")
    _stop_full = datetime.datetime.strptime(_stop_full_raw, _dt_fmt)
    _stop = _start + datetime.timedelta(days=smoke_test_days)
    assert _stop <= _stop_full, (
        f"smoke_test_days={smoke_test_days} runs past the model window "
        f"({_start} -> {_stop_full})"
    )

    # replace the value in place so the column padding and trailing comment survive
    for _k, _ln in enumerate(_lines):
        if _ln.startswith("StopDateTime"):
            _lines[_k] = _ln.replace(_stop_full_raw, _stop.strftime(_dt_fmt), 1)
    dflow_config.write_text("".join(_lines))
    print(f"SMOKE TEST     : StopDateTime {_stop_full} -> {_stop} "
          f"({smoke_test_days:g} d, {int(smoke_test_days * 86400 / dflowfm_dtuser // dflow_per_mf)} coupling steps)")

In [ ]:
# make the D-Flow FM dll discoverable, then initialize via BMI
os.environ["PATH"] = str(dflow_dirpath) + os.pathsep + os.environ["PATH"]
assert (dflow_dirpath / "dflowfm.dll").is_file(), dflow_dirpath

dflowfm = BMIWrapper(engine="dflowfm", configfile=str(dflow_config))
dflowfm.initialize()   # NOTE: this changes the working directory to the D-Flow run dir

### Grid variables and the source/sink exchange array

In [ ]:
ndxi = int(dflowfm.get_var("ndxi"))
ndx = int(dflowfm.get_var("ndx"))
qext = np.zeros(ndx)          # groundwater exchange pushed into D-Flow FM
qext_cum = np.zeros(ndx)
vextcum = dflowfm.get_var("vextcum")

def _read_mdu(path):
    cfg = {}
    for line in open(path):
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = [s.strip() for s in line.split("=", 1)]
        cfg[k.lower()] = v
    return cfg

_mdu = _read_mdu(dflow_config)

# RefDate is the epoch the .bc/.tim "seconds since" values are measured from -- it is
# NOT the start of the computation (StartDateTime/StopDateTime set that).
dflow_ref_date = datetime.datetime.strptime(_mdu["refdate"].split()[0], "%Y%m%d")
dflow_t0 = float(dflowfm.get_start_time())     # seconds since RefDate
dflow_t1 = float(dflowfm.get_end_time())
dflow_start_date = dflow_ref_date + datetime.timedelta(seconds=dflow_t0)
dflow_end_date = dflow_ref_date + datetime.timedelta(seconds=dflow_t1)

print(f"D-Flow refdate : {dflow_ref_date}  (ndxi={ndxi}, ndx={ndx})")
print(f"D-Flow run     : {dflow_start_date} -> {dflow_end_date}")
print(f"               : {dflow_t0:,.0f} -> {dflow_t1:,.0f} s since refdate "
      f"({(dflow_t1 - dflow_t0)/86400.:.1f} d)")

## Initialize SWMM

In [ ]:
# D-Flow FM's initialize() has ALREADY moved the working directory to the D-Flow run
# dir (see the cell above), and it is not restored until after the run loop. A bare
# "../swmm/..." therefore resolves against the run dir -- e.g.
# dflow-fm/coarse/swmm/gp/gp_sewer.inp -- so anchor it to the notebook directory.
swmm_path = (nb_dir / f"../swmm/{domain}/{swmm_inp_name}").resolve()
print(swmm_path, swmm_path.is_file())
assert swmm_path.is_file(), swmm_path

for ext in (".out", ".rpt"):        # remove stale SWMM outputs
    p = swmm_path.with_suffix(ext)
    if p.is_file():
        p.unlink()

# NOTE: only ONE pyswmm.Simulation may be open on a given .inp at a time. Opening a
# second handle just to read the dates -- even inside a `with` block -- closes the
# SWMM library out from under this one and leaves a truncated header-only .rpt.
swmm_sim = pyswmm.Simulation(str(swmm_path))
swmm_start_date, swmm_end_date = swmm_sim.start_time, swmm_sim.end_time
print("SWMM:", swmm_start_date, "->", swmm_end_date)

# Fail loudly (and before D-Flow has been advanced) if the resolved connection
# junctions are not present in the model we are actually simulating.
swmm_node_ids = {n.nodeid for n in pyswmm.Nodes(swmm_sim)}
missing = [j for j in junctions if j not in swmm_node_ids]
assert not missing, (
    f"{len(missing)} of {len(junctions)} connection junctions are absent from "
    f"{swmm_path.name} (first few: {missing[:5]}). intersect_points_grid resolves "
    f"junctions against ../swmm/{domain}/{domain}_sewer.inp, so swmm_inp_name must "
    "refer to that same model."
)

swmm_nodes = {j: pyswmm.Nodes(swmm_sim)[j] for j in junctions}
swmm_sim.start()

# SWMM and D-Flow FM do NOT start at the same instant: gp_sewer.inp starts at
# 2010-01-01 00:00 while the coarse D-Flow window starts at 12:00. The coupling
# loop advances SWMM by one MODFLOW time step per coupling step, i.e. by exactly
# the D-Flow run length, so an unaligned SWMM finishes that same offset EARLY --
# 12.25 h short for the coarse grid. The regenerated tracer .bc then stops before
# the D-Flow window closes, and iterating on that file bakes the hole in for good.
#
# Fast-forward SWMM to the D-Flow start so the two clocks agree. SWMM still routes
# internally at its own ROUTING_STEP during the catch-up, so this is spin-up that
# D-Flow simply never sees.
if swmm_start_date < dflow_start_date:
    lead_s = int((dflow_start_date - swmm_start_date).total_seconds())
    swmm_sim.step_advance(lead_s)
    next(swmm_sim)
    print(f"advanced SWMM {lead_s / 3600.0:.2f} h to align with the D-Flow start "
          f"({swmm_start_date} -> {swmm_sim.current_time})")
elif swmm_start_date > dflow_start_date:
    print(f"WARNING: SWMM starts {swmm_start_date} AFTER D-Flow {dflow_start_date}; "
          "the first part of the D-Flow window has no sewer forcing")

# the SWMM run has to span the D-Flow window or the coupling loop stops early
if swmm_end_date < dflow_end_date:
    print(
        f"WARNING: SWMM ends {swmm_end_date} but D-Flow ends {dflow_end_date} - "
        "the coupling loop will break on StopIteration before D-Flow finishes"
    )

## Initialize MODFLOW via the MODFLOW API

In [ ]:
mf6 = ModflowApi(str(libmf6), working_directory=str(mf_run_path))
mf6.initialize()

apisim = ApiSimulation.load(mf6)
apiml = apisim.get_model()
sewer_flow = apiml.get_package("swmm")           # the SWMM well package handle
swmm_dtype = [("nodelist", "O"), ("q", float)]

### MODFLOW variable pointers (GHB bhead/cond, CHD head, boundary flows)

In [ ]:
ghb_bhead_ptr = mf6.get_value_ptr(mf6.get_var_address("BHEAD", "GWF", "GHB"))
ghb_cond_ptr = mf6.get_value_ptr(mf6.get_var_address("COND", "GWF", "GHB"))
ghb_flow_tag = mf6.get_var_address("SIMVALS", "GWF", "GHB")

chd_head_ptr = mf6.get_value_ptr(mf6.get_var_address("HEAD", "GWF", "chd_surface"))
chd_flow_tag = mf6.get_var_address("SIMVALS", "GWF", "chd_surface")

### Result dictionaries (keyed by coupling step)

In [ ]:
ghb_elev_dict, ghb_cond_dict, chd_elev_dict = {}, {}, {}
qext_dict, swmm_q_dict = {}, {}
# [inactive MODFLOW cells, dry pipes, no-exchange pipes] per coupling step
pipe_state_dict = {}


## Coupling exchange functions

In [ ]:
def update_mf(key, s, d):
    """D-Flow FM water level (s) -> MODFLOW GHB bhead/cond and CHD head."""
    mask = d == 0.0
    s = np.where(mask, 0.0, s)      # never write back into D-Flow's own s1 array
    mult = np.full(d.shape, 1.0)
    mult[mask] = 0.0

    # Series.to_numpy() hands back a VIEW into ghb_data0/chd_data0. Without .copy()
    # the "template" is overwritten on the first coupling step, and because the cond
    # update is multiplicative, GHB conductance then decays toward zero every step.
    ghb_head = ghb_data0["bhead"].to_numpy().copy()
    ghb_head[ghbmask] = dflow2mfghb.dot(s)[ghbmask] * m2ft
    ghb_cond = ghb_data0["cond"].to_numpy().copy()
    ghb_cond[ghbmask] = ghb_cond[ghbmask] * dflow2mfghb.dot(mult)[ghbmask]

    chd_head = chd_data0["head"].to_numpy().copy()
    chd_head[chdmask] = dflow2mfchd.dot(s)[chdmask] * m2ft

    # trap NaN/Inf before it crosses the API into MODFLOW
    for nm, arr in (("ghb_head", ghb_head), ("ghb_cond", ghb_cond), ("chd_head", chd_head)):
        bad = ~np.isfinite(arr)
        assert not bad.any(), f"step {key}: {int(bad.sum())} non-finite values in {nm}"

    ghb_bhead_ptr[:] = ghb_head[:]
    ghb_cond_ptr[:] = ghb_cond[:]
    chd_head_ptr[:] = chd_head[:]

    ghb_elev_dict[key] = ghb_head.copy()
    ghb_cond_dict[key] = ghb_cond.copy()
    chd_elev_dict[key] = chd_head.copy()

In [ ]:
def update_dflow(key, d):
    """MODFLOW boundary flows -> D-Flow FM groundwater exchange (qext)."""
    ghb_flow = -mf6.get_value(ghb_flow_tag) * cfd2cms
    dflow_qext_ghb = ghb2qext.dot(ghb_flow)
    dflow_qext_ghb[d == 0.0] = 0.0

    chd_flow = -mf6.get_value(chd_flow_tag) * cfd2cms
    dflow_qext_chd = chd2qext.dot(chd_flow)
    dflow_qext_chd[d == 0.0] = 0.0

    dflow_qext = dflow_qext_ghb + dflow_qext_chd

    # trap NaN/Inf before it crosses the BMI into D-Flow FM
    bad = ~np.isfinite(dflow_qext)
    assert not bad.any(), f"step {key}: {int(bad.sum())} non-finite values in qext"

    qext_cum[:ndxi] += dflow_qext[:ndxi]
    qext[:ndxi] = dflow_qext[:ndxi]
    dflowfm.set_var("qext", qext)

    qext_dict[key] = qext[:ndxi].copy()

In [ ]:
def update_swmm(key):
    """Sewer <-> aquifer exchange, driven by the water surface INSIDE the pipe.

    Two regimes, switched on whether the water table reaches the pipe:

    CONNECTED  (head > invert)
        The ground around the pipe is saturated, so the driving head is the
        groundwater head against the water surface in the pipe:
            pot = head - (invert + depth)
        Positive means infiltration into the pipe, negative exfiltration into
        saturated ground. A dry pipe reduces this to head - invert, i.e. pure
        infiltration, which is right.

    DISCONNECTED  (head <= invert)
        The water table has fallen below the pipe, so the ground beneath it is
        unsaturated and the pipe free-drains. The gradient is then set by the
        depth of water in the pipe itself and does NOT depend on how far below
        the water table has dropped -- the same treatment a disconnected stream
        or a MODFLOW DRN gets:
            pot = -depth
        An empty pipe gives pot = 0, so a dry pipe over a deep water table
        exchanges nothing, which the previous head-minus-invert form got wrong:
        it produced exfiltration from an empty pipe that grew without limit as
        the water table fell.

    Conductance is per-junction when pipe_leakance is set (area-based), otherwise
    the legacy per-connection constant. Exfiltration uses a smaller value than
    infiltration either way.
    """
    heads = apiml.X
    mf6_spd = []
    n_inactive = n_disconnected = n_pipe_dry = 0
    for name, node in swmm_nodes.items():
        lay, row, col = mf6_cells[name]
        head = heads[lay, row, col]     # read the head in the well's OWN layer
        # A dry or inactive cell comes back as HDRY (-1e30); using it unguarded sends
        # a ~1e26 flux into both SWMM and MODFLOW.
        if (not np.isfinite(head)) or head <= 0.5 * HDRY:
            n_inactive += 1
            Q = 0.0
        else:
            invert_ft = swmm_inverts[name] * m2ft
            depth_ft = float(node.depth) * m2ft          # pyswmm depth is above invert
            if depth_ft <= PIPE_DEPTH_MIN:
                depth_ft = 0.0
                n_pipe_dry += 1

            if head > invert_ft:
                pot = head - (invert_ft + depth_ft)      # connected
            else:
                pot = -depth_ft                          # disconnected: free drainage
                n_disconnected += 1

            if pot == 0.0:
                Q = 0.0
            else:
                if swmm_conductance is None:
                    cond = (0.001 if pot > 0.0 else 0.0002) * conductance_scale
                else:
                    cond = swmm_conductance[name]
                    if pot < 0.0:
                        cond *= exfiltration_ratio
                Q = pot * cond
        node.generated_inflow(Q * cfd2cms)          # into SWMM (CMS)
        mf6_spd.append(((lay, row, col), -Q))       # out of MODFLOW

    if n_inactive:
        print(f"\n  step {key}: {n_inactive} dry/non-finite MODFLOW cells -> Q set to 0")

    mf6_spd = np.array(mf6_spd, dtype=swmm_dtype)
    assert np.isfinite(mf6_spd["q"]).all(), f"step {key}: non-finite SWMM well flux"

    sewer_flow.stress_period_data.values = mf6_spd
    swmm_q_dict[key] = mf6_spd["q"].copy()
    pipe_state_dict[key] = np.array(
        [n_inactive, n_disconnected, n_pipe_dry], dtype=np.int32)

## Run the coupled models

D-Flow FM advances every `DtUser`; every `dflow_per_mf` D-Flow steps the three
models exchange (one MODFLOW step + one SWMM step). This assumes the three models
share a start date; see the `_BNB` notebook for the date-staggered warm-up.

In [ ]:
idx, jdx = 0, 0
t0 = time.perf_counter()
current_time = dflowfm.get_current_time()
end_time = dflowfm.get_end_time()

# progress is measured across the computation WINDOW: current_time is seconds since
# RefDate (2000-01-01), so current_time/end_time is ~99.9% on the very first step
# and tells you nothing.
window = end_time - dflow_t0

while current_time <= end_time:
    idx += 1
    dflowfm.update()
    current_time = dflowfm.get_current_time()
    elapsed = current_time - dflow_t0
    print(f"  {elapsed/86400.:7.3f} of {window/86400.:.2f} d  {elapsed/window:6.1%}  "
          f"step {jdx:05d}", end="\r")

    if idx == int(dflow_per_mf):
        # .copy() -- get_var() returns a view into D-Flow FM's own memory, and
        # update_mf masks these arrays before use.
        s = dflowfm.get_var("s1")[:ndxi].copy()
        d = dflowfm.get_var("hs")[:ndxi].copy()

        mf6.prepare_time_step(mf6.get_time_step())
        update_mf(str(jdx), s, d)
        mf6.do_time_step()
        mf6.finalize_time_step()
        update_dflow(str(jdx), d)

        update_swmm(str(jdx))
        swmm_sim.step_advance(int(mf6.get_time_step() * d2sec))
        try:
            swmm_sim.__next__()
        except StopIteration:
            print(f"\nSWMM ended at coupling step {jdx} "
                  f"({elapsed/86400.:.2f} d of {window/86400.:.2f} d)")
            break

        idx = 0
        jdx += 1

    if current_time >= end_time:
        break

vextcum = dflowfm.get_var("vextcum")
print(f"\nrun time: {(time.perf_counter() - t0) / 60.0:.2f} min ({jdx} coupling steps)")

### Finalize the three models

In [ ]:
mf6.finalize()
swmm_sim.terminate_simulation()
swmm_sim.report()
swmm_sim.close()
dflowfm.finalize()

os.chdir(nb_dir)   # D-Flow init changed the cwd; restore it for the save/regen cells

## Save MODFLOW results (scenario-named)

Exchange arrays as compressed `.npz` and the MODFLOW head/concentration output,
into the scenario results directory for the step3 plotting notebooks.

In [ ]:
np.savez_compressed(results_ws / "ghb_elev.npz", **ghb_elev_dict)
np.savez_compressed(results_ws / "ghb_cond.npz", **ghb_cond_dict)
np.savez_compressed(results_ws / "chd_elev.npz", **chd_elev_dict)
np.savez_compressed(results_ws / "qext.npz", **qext_dict)
np.savez_compressed(results_ws / "swmm_q.npz", **swmm_q_dict)
np.savez_compressed(results_ws / "pipe_state.npz", **pipe_state_dict)

# Anything the OBS packages were told to write under outputs/ (the SWMM well obs).
for out in (mf_run_path / "outputs").glob("*"):
    shutil.copy2(out, results_ws / out.name)

# MF6 writes the remaining observation csvs and the listing files to the run
# directory ROOT, not outputs/, so copy those too -- step3_plot_compare_coastal_
# exchange reads gwf.ghb.obs.csv / gwf.chd.obs.csv from here.
#
# The head, budget and concentration files are deliberately NOT copied: gwf.cbc
# is ~184 MB and gwf.hds / gwt.ucn ~32 MB each, so duplicating them would cost
# ~250 MB per scenario. step3 opens those in place via
# liss_settings.get_modflow_run_path(), which resolves the same directory.
for pattern in ("*.obs.csv", "*.obs.output.csv", "*.lst"):
    for out in mf_run_path.glob(pattern):
        shutil.copy2(out, results_ws / out.name)

print("saved MODFLOW results ->", results_ws)
print(f"  heads/budget/concentration left in place: {mf_run_path}")

## Save SWMM results (scenario-named)

In [ ]:
for ext in (".out", ".rpt"):
    p = swmm_path.with_suffix(ext)
    if p.is_file():
        shutil.copy2(p, results_ws / f"swmm{ext}")
print("saved SWMM results ->", results_ws)

## Save the D-Flow FM tracer results (not the whole map file)

Extract only the sewage tracer field (`mesh2d_sewage`) plus the mesh geometry,
and write a compact scenario NetCDF &mdash; avoiding the multi-hundred-MB full
`FlowFM_map.nc`.

In [ ]:
map_path = dflow_working / "output" / "FlowFM_map.nc"
tracer_out = results_ws / "dflow_tracer.nc"

# Keep the fields the step3 notebooks actually use:
#   mesh2d_sewage     -> step3_plot_tracer
#   mesh2d_waterdepth -> wet/dry masking in both
#   mesh2d_s1         -> water level, step3_plot_modflow_results
# The full map carries ~30 variables and is ~11.5 GB for the coarse 89-day run,
# so trimming keeps the scenario results usable on their own.
tracer_vars = ("mesh2d_sewage", "mesh2d_waterdepth", "mesh2d_s1")

if map_path.is_file():
    ds = xugrid.open_dataset(map_path)
    keep = [v for v in tracer_vars if v in ds]
    # MUST be .ugrid.to_netcdf() -- a plain .to_netcdf() writes the data and the
    # face coordinates but DROPS the UGRID topology variable, and the result then
    # fails to reopen with "The file or object does not contain UGRID conventions
    # data", which is exactly what the step3 notebooks need to build the mesh.
    ds[keep].ugrid.to_netcdf(tracer_out)
    ds.close()

    # verify it can actually be reopened as UGRID before the run dir is reused
    _chk = xugrid.open_dataset(tracer_out)
    _ncell = _chk["mesh2d_nFaces"].size
    _chk.close()
    print(f"saved D-Flow tracer -> {tracer_out}")
    print(f"  {keep}")
    print(f"  {_ncell} cells, {tracer_out.stat().st_size / 1e9:,.2f} GB "
          f"(full map was {map_path.stat().st_size / 1e9:,.1f} GB)")
else:
    print("no D-Flow map file found at", map_path)

## Regenerate the sewer tracer `.bc` from this run's SWMM results

Rebuild `Sewer_sourcesink_n{n_connections}__{tag}.bc` from the SWMM outflow so a
subsequent run uses updated sewer forcing. Reuses `write_bc` from
`data/GP/update_files.py` (no duplicated code).

> **TODO / verify:** confirm the outfall node id that drives the D-Flow FM
> source/sink, the tracer concentration, and the SWMM output units/reference
> time before relying on this.

In [ ]:
# reuse the .bc writer from data/GP/update_files.py
sys.path.append(str((nb_dir / f"../data/{domain.upper()}").resolve()))
from update_files import write_bc, source_id, ref_time

tracer_conc = 1000.0                 # kg/m3 sewage tracer (see update_files.py)

ref = datetime.datetime.strptime(ref_time, "%Y-%m-%d %H:%M:%S")
assert ref == dflow_ref_date, (
    f"update_files.ref_time ({ref}) must match the D-Flow RefDate ({dflow_ref_date}); "
    "otherwise the .bc time column is offset from the model clock"
)


def read_bc(path):
    """Read a Sewer_sourcesink .bc back as (times, discharge, tracer).

    Mirrors write_bc: a [General] header then two [Forcing] blocks sharing one
    time column -- sourcesink_discharge first, then sourcesink_tracersewageDelta.
    """
    blocks, cur = [], None
    for line in pl.Path(path).read_text().splitlines():
        s = line.strip()
        if s.startswith("[Forcing]"):
            cur = []
            blocks.append(cur)
            continue
        if cur is None or not s or "=" in s or s.startswith("["):
            continue
        parts = s.split()
        if len(parts) >= 2:
            try:
                cur.append((float(parts[0]), float(parts[1])))
            except ValueError:
                continue
    assert len(blocks) == 2, f"{path}: expected 2 [Forcing] blocks, found {len(blocks)}"
    t0 = [t for t, _ in blocks[0]]
    t1 = [t for t, _ in blocks[1]]
    assert t0 == t1, f"{path}: the two [Forcing] blocks have different time columns"
    return t0, [v for _, v in blocks[0]], [v for _, v in blocks[1]]


# gp_sewer.inp declares FLOW_UNITS CMS, so TOTAL_INFLOW is already m3/s and matches
# the unit write_bc declares -- no conversion.
with pyswmm.Output(str(swmm_path.with_suffix(".out"))) as out:
    series = out.node_series(outfall_node, NodeAttribute.TOTAL_INFLOW)
    new_times = [(t - ref).total_seconds() for t in series.keys()]
    new_q = list(series.values())

# Splice rather than replace: this run only simulated the D-Flow window, so keep
# whatever the previous file held OUTSIDE that window. Otherwise each iteration
# would truncate the series to the window and a later, longer run would have no
# sewer forcing beyond it.
old_times, old_q, old_tr = read_bc(tracer_bc_src)
lo, hi = dflow_t0, dflow_t1

merged = [(t, q) for t, q in zip(old_times, old_q) if t < lo or t > hi]
inside = [(t, q) for t, q in zip(new_times, new_q) if lo <= t <= hi]
merged.extend(inside)
merged.sort(key=lambda r: r[0])

times = [t for t, _ in merged]
discharge = [q for _, q in merged]
tracer = [tracer_conc] * len(times)

assert times == sorted(times), "spliced time column is not monotonic"
assert len(set(times)) == len(times), "spliced time column has duplicates"

# The authoritative copy lives in data/<DOMAIN>/ so the next run of THIS scenario
# picks it up (see the tracer-forcing cell above). A copy also goes to the results
# directory as the record of what this particular run produced.
bc_out = tracer_bc_scenario
write_bc(bc_out, source_id, times, discharge, tracer, ref_time=ref_time)
shutil.copy2(bc_out, results_ws / bc_out.name)

print(f"regenerated tracer bc -> {bc_out}")
print(f"  copy for the record -> {results_ws / bc_out.name}")
print(f"  outfall {outfall_node}; {len(inside):,} new points inside the window, "
      f"{len(merged) - len(inside):,} kept from {tracer_bc_src.name} outside it")
print(f"  window        {lo:>15,.0f} -> {hi:>15,.0f} s since {ref_time}")
print(f"  file spans    {times[0]:>15,.0f} -> {times[-1]:>15,.0f} s "
      f"({(times[-1] - times[0]) / 86400.0:.1f} d, {len(times):,} points)")
if inside:
    print(f"  new data      {inside[0][0]:>15,.0f} -> {inside[-1][0]:>15,.0f} s")
if times[0] > dflow_t0:
    print(f"  WARNING: series starts {(times[0] - dflow_t0) / 86400.0:.2f} d after the "
          "D-Flow window opens")
if times[-1] < dflow_t1:
    print(f"  WARNING: series ends {(dflow_t1 - times[-1]) / 86400.0:.2f} d before the "
          "D-Flow window closes")
if not inside:
    print("  WARNING: no new SWMM points fell inside the window - nothing was updated")

---
### Notes / things to verify before a production run
- **Mapping weights** must exist for the chosen `resolution` (`step1a`/`step1b`).
- **Tracer bc** `Sewer_sourcesink_n{n_connections}__{tag}.bc` must exist for the
  actual connection count + coupling tag (regenerate via `update_files.py`).
- **SWMM inflow coefficients** in `update_swmm` (`0.001` / `0.0002`) are
  rescaled by `n_original / n_connections` (`n_original = 12`) so the total
  SWMM&harr;MODFLOW conductance is invariant to the number of connections.
- **bc regeneration** node id / units are scaffolded &mdash; confirm against
  `data/GP/update_files.py` and the SWMM model.
- The coupled run is long; run it outside this authoring session.